# Reading Data

In [3]:
import pandas as pd
import re
import os

"""
Load 14 multi-omics datasets from parquet files for cancer cell line analysis.

Loads HPA RNA, DepMap expression, GEO expression, proteomics, fusions, 
mutations, cellosaurus metadata, metabolomics, miRNA, and global signatures data.

Constants:
    PARQUET (str): Base path '/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet'

Returns (loaded DataFrames):
    hpa_rna, depmap_expr, geo_expr, proteomics, fusions, mutations, 
    cellosaurus, depmap_profiles, sample_info, geo_info, hpa_desc,
    metabolomics, mirna, signatures : pd.DataFrame

Dependencies:
    pandas, re, os
"""

PARQUET = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet"

# Load all 14 from parquet
hpa_rna         = pd.read_parquet(f"{PARQUET}/1_4_hpa_rna_celline.parquet")
depmap_expr     = pd.read_parquet(f"{PARQUET}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.parquet")
geo_expr        = pd.read_parquet(f"{PARQUET}/3_GEOexpression.parquet")
proteomics      = pd.read_parquet(f"{PARQUET}/4_Harmonized_MS_CCLE_Gygi_subsetted.parquet")
fusions         = pd.read_parquet(f"{PARQUET}/5_OmicsFusionFilteredSupplementary.parquet")
mutations       = pd.read_parquet(f"{PARQUET}/6_OmicsSomaticMutationsProfile.parquet")
cellosaurus     = pd.read_parquet(f"{PARQUET}/7_cellosaurus.parquet")
depmap_profiles = pd.read_parquet(f"{PARQUET}/8_DepMap_OmicsProfiles.parquet")
sample_info     = pd.read_parquet(f"{PARQUET}/9_DepMap_sample_info.parquet")
geo_info        = pd.read_parquet(f"{PARQUET}/10_GEOInfo.parquet")
hpa_desc        = pd.read_parquet(f"{PARQUET}/11_hpa_rna_celline_description.parquet")
metabolomics    = pd.read_parquet(f"{PARQUET}/12_CCLE_metabolomics_20190502.parquet")
mirna           = pd.read_parquet(f"{PARQUET}/13_CCLE_miRNA_20181103.parquet")
signatures      = pd.read_parquet(f"{PARQUET}/14_OmicsGlobalSignatures.parquet")

# Checking the Missing Data from Each File Through Each Column

In [4]:
def audit(df, name):
    """
    Audits a pandas DataFrame and returns a styled quality report.

    Args:
        df (pd.DataFrame): The DataFrame to audit.
        name (str): Display name for the dataset.

    Returns:
        pandas.io.formats.style.Styler: A styled DataFrame containing column-wise
            audit information including data types, missing percentages (color-coded),
            and sample values.

    The missing percentage color scheme:
        - 0%: Green (no missing values)
        - <10%: Yellow (low missingness)
        - 10-40%: Orange (moderate missingness)
        - >40%: Red (high missingness)

    Example:
        >>> df = pd.DataFrame({'A': [1, None, 3], 'B': ['x', 'y', 'z']})
        >>> audit(df, "Test Data")
    """
    print(f"  {name}  —  {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    rows = []
    for col in df.columns:
        n_missing  = df[col].isnull().sum()
        pct        = n_missing / len(df) * 100
        sample     = df[col].dropna().iloc[:2].tolist() if n_missing < len(df) else ["ALL NULL"]
        rows.append({
            "Column":        col,
            "Dtype":         str(df[col].dtype),
            "Missing %":     f"{pct:.1f}%",
            "Sample values": str(sample)[:80]
        })
    result = pd.DataFrame(rows)

    def colour(val):
        pct = float(val.strip('%'))
        if pct == 0:  return 'background-color:#d4edda;color:#155724'
        if pct < 10:  return 'background-color:#fff3cd;color:#856404'
        if pct < 40:  return 'background-color:#ffd699;color:#6d3a00'
        return              'background-color:#f8d7da;color:#721c24'

    return result.style.map(colour, subset=["Missing %"]).hide(axis="index")

In [7]:
audit(hpa_rna, "1. HPA RNA")

  1. HPA RNA  —  24,315,372 rows × 6 cols


Column,Dtype,Missing %,Sample values
Gene,object,0.0%,"['ENSG00000000003', 'ENSG00000000003']"
Gene name,object,0.0%,"['TSPAN6', 'TSPAN6']"
Cell line,object,0.0%,"['143B', '22Rv1']"
TPM,float64,0.0%,"[22.0, 2.8]"
pTPM,float64,0.0%,"[27.6, 3.6]"
nTPM,float64,0.0%,"[25.9, 2.7]"


In [6]:
audit(depmap_expr.iloc[:, :20], "2. DepMap Expression — first 20 gene cols")

  2. DepMap Expression — first 20 gene cols  —  1,495 rows × 20 cols


Column,Dtype,Missing %,Sample values
TSPAN6 (ENSG00000000003),float64,0.0%,"[4.33199177823206, 4.567423757707283]"
TNMD (ENSG00000000005),float64,0.0%,"[0.0, 0.5849625007211562]"
DPM1 (ENSG00000000419),float64,0.0%,"[7.364659971814762, 7.106641452324356]"
SCYL3 (ENSG00000000457),float64,0.0%,"[2.792855352362489, 2.543495883425772]"
C1orf112 (ENSG00000000460),float64,0.0%,"[4.471187460386985, 3.5046203924035524]"
FGR (ENSG00000000938),float64,0.0%,"[0.0285691521967709, 0.0]"
CFH (ENSG00000000971),float64,0.0%,"[1.2265085298086795, 0.1890338243900171]"
FUCA2 (ENSG00000001036),float64,0.0%,"[3.0443941193584534, 3.8135246892978105]"
GCLC (ENSG00000001084),float64,0.0%,"[6.5000052920931335, 4.221877081077034]"
NFYA (ENSG00000001167),float64,0.0%,"[4.739848102699328, 3.48155728070859]"


In [9]:
print("Index name:", depmap_expr.index.name)
print("Sample index values:", depmap_expr.index[:5].tolist())

Index name: index
Sample index values: ['PR-AdBjpG', 'PR-I2AzwG', 'PR-5ekAAC', 'PR-I21681', 'PR-i9DRP1']


In [11]:
audit(geo_expr.iloc[:, :15], "3. GEO Expression — first 15 cols")


  3. GEO Expression — first 15 cols  —  19,914 rows × 15 cols


Column,Dtype,Missing %,Sample values
Gene,object,0.0%,"['ENSG00000000003', 'ENSG00000000005']"
GSM101610,float64,0.0%,"[33.6156997680664, 40.9256820678711]"
GSM101615,float64,0.0%,"[553.249755859375, 31.3274059295654]"
GSM101616,float64,0.0%,"[540.452209472656, 33.9349670410156]"
GSM101667,float64,0.0%,"[599.43115234375, 34.2131233215332]"
GSM101668,float64,0.0%,"[625.242736816406, 32.4662857055664]"
GSM101671,float64,0.0%,"[400.554656982422, 36.2432327270508]"
GSM101672,float64,0.0%,"[412.99560546875, 37.9525108337402]"
GSM101673,float64,0.0%,"[427.403900146484, 34.6444282531738]"
GSM101674,float64,0.0%,"[461.12353515625, 34.2254104614258]"


In [14]:
audit(proteomics.iloc[:, :15], "4. Proteomics — first 15 cols")


  4. Proteomics — first 15 cols  —  375 rows × 15 cols


Column,Dtype,Missing %,Sample values
Unnamed: 0,object,0.0%,"['ACH-000849', 'ACH-000441']"
A0AV96 (RBM47),float64,0.0%,"[0.358566705, -1.1124099902]"
A0AVF1 (IFT56),float64,28.8%,"[-0.1720661669, 0.3394459393]"
A0AVG3 (TSNARE1),float64,83.2%,"[-2.0942353374, -0.4262699812]"
A0AVI4 (TMEM129),float64,78.4%,"[-0.1661644624, 0.0913671891]"
A0AVK6 (E2F8),float64,80.8%,"[-0.3864824182, 0.0043318875]"
A0AVT1 (UBA6),float64,0.0%,"[-0.4968418709, -0.3956330964]"
A0JLT2 (MED19),float64,16.5%,"[0.2676913493, -0.2627149509]"
A0JNW5 (BLTP3B),float64,4.8%,"[0.045753535, -0.414909128]"
A0MZ66 (SHTN1),float64,0.0%,"[0.0653424074, -0.3963701918]"


In [15]:
audit(fusions, "5. Fusions")


  5. Fusions  —  184,237 rows × 30 cols


Column,Dtype,Missing %,Sample values
Unnamed: 0,int64,0.0%,"[0, 1]"
SequencingID,object,0.0%,"['CDS-010xbm', 'CDS-010xbm']"
ModelID,object,0.0%,"['ACH-001113', 'ACH-001113']"
IsDefaultEntryForModel,object,0.0%,"['Yes', 'Yes']"
ModelConditionID,object,0.0%,"['MC-001113-k2lR', 'MC-001113-k2lR']"
IsDefaultEntryForMC,object,0.0%,"['Yes', 'Yes']"
CanonicalFusionName,object,0.0%,"['DLG1--SERPINI1', 'ADAM17--ITGB1BP1']"
gene1(ENS ID),object,0.0%,"['DLG1 (ENSG00000075711.21)', 'ADAM17 (ENSG00000151694.14)']"
gene2(ENS ID),object,0.0%,"['SERPINI1 (.)', 'ITGB1BP1 (ENSG00000119185.13)']"
TotalReadsInSample,int64,0.0%,"[47434547, 47434547]"


In [17]:
audit(mutations, "6. Mutations")


  6. Mutations  —  1,066,869 rows × 70 cols


Column,Dtype,Missing %,Sample values
Chrom,object,0.0%,"['chr1', 'chr1']"
Pos,int64,0.0%,"[818203, 851926]"
Ref,object,0.0%,"['G', 'G']"
Alt,object,0.0%,"['A', 'A']"
AF,float64,0.0%,"[0.24, 0.158]"
DP,int64,0.0%,"[27, 17]"
RefCount,int64,0.0%,"[21, 15]"
AltCount,int64,0.0%,"[6, 2]"
GT,object,0.0%,"['0/1', '0/1']"
PS,float64,93.6%,"[924510.0, 953259.0]"


In [8]:
"""
Verify missing value percentages for specific columns in the mutations dataset.

This script checks a predefined list of columns that were previously identified as
potentially having 100% missing values. It performs a detailed inspection to confirm
the actual missing value percentage and displays any non-null values if present.

Columns checked:
    - Brca1FuncScore: BRCA1 functional score
    - PharmgkbId: PharmGKB database identifier
    - DidaID: DIDA database identifier
    - DidaName: DIDA database name
    - GwasDisease: GWAS-associated disease
    - GwasPmID: GWAS PubMed ID
    - GtexGene: GTEx gene expression information

For each column, the script reports:
    - Total number of rows in the dataset
    - Count of null/missing values
    - Count of non-null values
    - Percentage of missing values (with 4 decimal precision)
    - Sample of up to 5 non-null values (if any exist)

Purpose:
    To validate data quality assertions and identify columns that can be safely
    dropped due to complete missingness, or to discover unexpected non-null values
    that may require further investigation.

Usage example:
    Run this code block after loading the mutations dataset to verify column completeness.

Expected output:
    If columns are truly 100% missing, output will show null_count = total rows,
    non_null = 0, and missing % = 100.0000%. No non-null values will be displayed.
    If any non-null values exist, they will be printed for inspection.
"""

cols_100 = ["Brca1FuncScore", "PharmgkbId", "DidaID", "DidaName", 
            "GwasDisease", "GwasPmID", "GtexGene"]

print("Checking supposedly 100% missing columns\n")

for col in cols_100:
    total      = len(mutations)
    null_count = mutations[col].isnull().sum()
    non_null   = total - null_count
    pct_missing = null_count / total * 100
    
    print(f"Column: {col}")
    print(f"  Total rows:     {total:,}")
    print(f"  Null count:     {null_count:,}")
    print(f"  Non-null count: {non_null:,}")
    print(f"  Missing %:      {pct_missing:.4f}%")
    
    if non_null > 0:
        print(f"  Non-null values:")
        print(f"  {mutations[col].dropna().head(5).tolist()}")
    print()

Checking supposedly 100% missing columns

Column: Brca1FuncScore
  Total rows:     1,066,869
  Null count:     1,066,835
  Non-null count: 34
  Missing %:      99.9968%
  Non-null values:
  [-2.763438152, -0.2882635071, -2.734314743, -1.899596881, -0.1421652642]

Column: PharmgkbId
  Total rows:     1,066,869
  Null count:     1,066,839
  Non-null count: 30
  Missing %:      99.9972%
  Non-null values:
  ['PA166154663', 'PA166154663', 'PA166154663', 'PA166154663', 'PA166154663']

Column: DidaID
  Total rows:     1,066,869
  Null count:     1,066,855
  Non-null count: 14
  Missing %:      99.9987%
  Non-null values:
  ['dd227', 'dd172', 'dd032', 'dd227', 'dd092']

Column: DidaName
  Total rows:     1,066,869
  Null count:     1,066,855
  Non-null count: 14
  Missing %:      99.9987%
  Non-null values:
  ['CANDLE syndrome', 'MODY', 'Familial dysfibrinogenemia', 'CANDLE syndrome', 'Bardet-Biedl syndrome']

Column: GwasDisease
  Total rows:     1,066,869
  Null count:     1,066,517
  Non-n

In [18]:
audit(cellosaurus, "7. Cellosaurus")


  7. Cellosaurus  —  152,231 rows × 17 cols


Column,Dtype,Missing %,Sample values
Identifier (cell line name),object,0.0%,"['#132 PC3-1-SC-E8', '#132 PL12 SC-D1']"
Accession (CVCL_xxxx),object,0.0%,"['CVCL_B0T9', 'CVCL_B0T8']"
Secondary accession number(s),object,99.6%,"['CVCL_A596', 'CVCL_7643']"
Synonyms,object,52.4%,"['Z48-5MG-70', 'Z48-5MG-63']"
Cross-references,object,1.2%,"['Wikidata; Q108819335', 'Wikidata; Q108819336']"
References identifiers,object,45.4%,"['Patent=EP0501779A1;', 'Patent=EP0501779A1;']"
Web pages,object,92.4%,['http://pathology.ucla.edu/workfiles/360cx.pdf || http://pathology.ucla.edu/wor
Comments,object,1.1%,['Group: Patented cell line. || Registration: International Depositary Authority
STR profile data,object,94.3%,"['Source(s): TKG || Amelogenin: X || CSF1PO: 12 || D13S317: 11 || D16S539: 9,11"
Diseases,object,53.4%,"['NCIt; C21619; Mouse mesothelioma', 'NCIt; C4878; Lung carcinoma']"


In [19]:
audit(depmap_profiles, "8. DepMap Omics Profiles")


  8. DepMap Omics Profiles  —  3,830 rows × 5 cols


Column,Dtype,Missing %,Sample values
ProfileID,object,0.0%,"['PR-00UtU3', 'PR-01r7OM']"
ModelCondition,object,0.0%,"['MC-001131-kkJv', 'MC-000957-Yckn']"
ModelID,object,0.0%,"['ACH-001131', 'ACH-000957']"
Datatype,object,0.0%,"['wgs', 'rna']"
WESKit,object,51.4%,"['ICE', 'AGILENT']"


In [20]:
audit(sample_info, "9. Sample Info")


  9. Sample Info  —  1,840 rows × 29 cols


Column,Dtype,Missing %,Sample values
DepMap_ID,object,0.0%,"['ACH-000016', 'ACH-000032']"
cell_line_name,object,5.0%,"['SLR 21', 'MHH-CALL-3']"
stripped_cell_line_name,object,0.1%,"['SLR21', 'MHHCALL3']"
CCLE_Name,object,0.2%,"['SLR21_KIDNEY', 'MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE']"
alias,object,93.9%,"['MUTZ3', 'TM87']"
COSMICID,float64,46.7%,"[905951.0, 1290907.0]"
sex,object,5.5%,"['Female', 'Female']"
source,object,2.6%,"['Academic lab', 'DSMZ']"
RRID,object,1.2%,"['CVCL_V607', 'CVCL_0089']"
WTSI_Master_Cell_ID,float64,46.7%,"[1835.0, 1036.0]"


In [21]:
audit(geo_info, "10. GEO Info")


  10. GEO Info  —  3,267 rows × 23 cols


Column,Dtype,Missing %,Sample values
Geo_accession,object,0.0%,"['GSM101610', 'GSM101615']"
CEL_file_names,object,0.0%,"['GSM101610', 'GSM101615']"
title,object,28.7%,"['786O_1a', '786O_1b']"
status,object,28.7%,"['Public on Oct 10 2012', 'Public on Oct 10 2012']"
submission_date,object,28.7%,"['Oct 10 2012', 'Oct 10 2012']"
last_update_date,object,28.7%,"['Oct 10 2012', 'Oct 10 2012']"
type,object,28.7%,"['RNA', 'RNA']"
channel_count,float64,28.7%,"[1.0, 1.0]"
source_name_ch1,object,28.7%,"['786-O', '786-O']"
organism_ch1,object,28.7%,"['Homo sapiens', 'Homo sapiens']"


In [22]:
audit(hpa_desc, "11. HPA Descriptions")


  11. HPA Descriptions  —  1,206 rows × 7 cols


Column,Dtype,Missing %,Sample values
Cell line,object,0.0%,"['143B', '22Rv1']"
Disease,object,0.0%,"['Bone cancer', 'Prostate cancer']"
Disease subtype,object,13.2%,"['Osteosarcoma', 'Adenocarcinoma']"
Cellosaurus ID,object,0.7%,"['CVCL_2270', 'CVCL_1045']"
Patient,object,8.7%,"['13', 'Male']"
Primary/Metastasis,object,28.1%,"['Primary', 'primary']"
Sample collection site,object,5.6%,"['bone', 'prostate']"


In [23]:
audit(metabolomics.iloc[:, :15], "12. Metabolomics — first 15 cols")


  12. Metabolomics — first 15 cols  —  928 rows × 15 cols


Column,Dtype,Missing %,Sample values
CCLE_ID,object,0.0%,"['DMS53_LUNG', 'SW1116_LARGE_INTESTINE']"
DepMap_ID,object,0.1%,"['ACH-000698', 'ACH-000489']"
2-aminoadipate,float64,0.0%,"[6.1127272, 5.5774126]"
3-phosphoglycerate,float64,0.0%,"[6.0341979, 5.7270453]"
alpha-glycerophosphate,float64,0.0%,"[5.8968963, 5.1114679]"
4-pyridoxate,float64,0.0%,"[6.0005322, 6.07325]"
aconitate,float64,0.0%,"[5.5136185, 5.8024939]"
adenine,float64,0.0%,"[5.8685293, 5.8244732]"
adipate,float64,0.0%,"[5.9771769, 5.8888208]"
alpha-ketoglutarate,float64,0.0%,"[5.6930736, 5.7683788]"


In [24]:
audit(mirna.iloc[:, :10], "13. miRNA — first 10 cols")


  13. miRNA — first 10 cols  —  734 rows × 10 cols


Column,Dtype,Missing %,Sample values
Name,object,0.0%,"['nmiR00001.1', 'nmiR00002.1']"
Description,object,0.0%,"['hsa-let-7a', 'hsa-let-7b']"
DMS53_LUNG,float64,0.0%,"[4362.58, 187.44]"
SW1116_LARGE_INTESTINE,float64,0.0%,"[5191.5, 868.22]"
NCIH1694_LUNG,float64,0.0%,"[24991.05, 5066.09]"
P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0.0%,"[3253.83, 74.21]"
HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0.0%,"[225.28, 35.74]"
UMUC3_URINARY_TRACT,float64,0.0%,"[35051.17, 1014.65]"
HOS_BONE,float64,0.0%,"[5706.63, 1211.27]"
HUNS1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,0.0%,"[1821.46, 22.54]"


In [25]:
audit(signatures, "14. Global Signatures")


  14. Global Signatures  —  3,021 rows × 12 cols


Column,Dtype,Missing %,Sample values
Unnamed: 0,int64,0.0%,"[0, 1]"
SequencingID,object,0.0%,"['CDS-00Nrci', 'CDS-051xn7']"
ModelID,object,0.0%,"['ACH-000839', 'ACH-000041']"
ModelConditionID,object,0.0%,"['MC-000839-krru', 'MC-000041-uPBf']"
IsDefaultEntryForModel,object,0.0%,"['Yes', 'Yes']"
IsDefaultEntryForMC,object,0.0%,"['Yes', 'Yes']"
MSIScore,float64,0.0%,"[3.68, 2.21]"
LoHFraction,float64,14.5%,"[0.107442983578596, 0.130088744871113]"
WGD,float64,14.5%,"[1.0, 1.0]"
CIN,float64,14.5%,"[0.502634318247591, 0.523865467657356]"


# Checking Overlap of Data Before Cleaning any Columns of any Table

In [ ]:
import re, pandas as pd

"""
Multi-omics dataset integration analysis: Gene and cell line overlap assessment.

This script performs systematic overlap analysis between multiple omics datasets
(DepMap, HPA, GEO, fusions, mutations) to identify common entities for integration.
It analyzes both gene-level (ENSG identifiers) and cell line-level identifiers,
providing critical information for multi-omics data harmonization.

Analysis Components:
    1. Gene ID Extraction & Standardization
        - DepMap: Extract ENSG IDs from SYMBOL (ENSG...) or bare ENSG columns
        - HPA/GEO: Use bare ENSG IDs from Gene column
        - Fusions: Strip version suffixes (e.g., ENSG00000075711.21 → ENSG00000075711)
        - Mutations: Use EnsemblGeneID column
    
    2. Gene Overlap Analysis
        - Individual dataset gene counts
        - Pairwise intersections (DepMap∩HPA, DepMap∩GEO, HPA∩GEO)
        - Triple intersection (DepMap∩HPA∩GEO) → high-confidence gene set
        - Dataset-specific genes (in only one dataset)
        - Coverage of fusion and mutation genes by DepMap
    
    3. Cell Line ID Resolution & Overlap
        - Resolve ProfileIDs to ModelIDs using depmap_profiles mapping
        - Handle different datatypes (RNA, WES, WGS)
        - Compare each dataset against master DepMap_ID list from sample_info
        - Calculate match percentages and orphaned counts
    
    4. HPA Cell Line Name Matching
        - Exact matching vs sample_info cell_line_name
        - Fuzzy matching using stripped_cell_line_name
        - Unmatched names requiring Cellosaurus resolution
    
    5. GEO Sample Coverage
        - Identify GSM columns in expression data
        - Map to GEO Info metadata
        - Check for Cellosaurus_ID and cell_line availability

Key Data Relationships:
    - Gene-level: ENSG identifiers as universal keys
    - Cell line-level: DepMap_ID (ACH-XXXXX) as primary key
    - Alternative IDs: RRID, Cellosaurus ID, CVCL IDs

Dependencies:
    - pandas: Data manipulation
    - re: Regular expressions for ENSG extraction and version stripping

Usage:
    Run this script after loading all 14 datasets to understand:
        1. Which genes can be integrated across platforms
        2. Which cell lines have complete multi-omics data
        3. Which datasets need additional ID mapping
        4. Quality of sample annotations and coverage

Output Interpretation:
    - High-confidence gene set: DepMap ∩ HPA ∩ GEO intersection
    - Cell line orphan rates indicate data completeness issues
    - Unmatched HPA names require Cellosaurus lookup
    - Unmapped GSMs should be excluded from analysis
"""


# ── 1. extract canonical IDs ──────────────────────────────────────────

# DepMap columns: "SYMBOL (ENSG...)" or bare "ENSG..."
depmap_ensg = set()
for col in depmap_expr.columns:
    m = re.search(r"(ENSG\d+)", col)
    if m: depmap_ensg.add(m.group(1))

# HPA: Gene column already bare ENSG
hpa_ensg = set(hpa_rna["Gene"].unique())

# GEO: Gene column already bare ENSG
geo_ensg = set(geo_expr["Gene"].unique())

# Fusions: strip version suffix e.g. ENSG00000075711.21
def strip_ver(val):
    m = re.search(r"(ENSG\d+)", str(val)) if pd.notna(val) else None
    return m.group(1) if m else None

fusion_ensg = set(filter(None,
    fusions["gene1(ENS ID)"].map(strip_ver).tolist() +
    fusions["gene2(ENS ID)"].map(strip_ver).tolist()
))

mutation_ensg = set(mutations["EnsemblGeneID"].dropna().unique())

# ── 2. gene overlap ───────────────────────────────────────────────────
print("=" * 55)
print("  GENE ID OVERLAP")
print("=" * 55)
print(f"  DepMap genes:           {len(depmap_ensg):>7,}")
print(f"  HPA genes:              {len(hpa_ensg):>7,}")
print(f"  GEO genes:              {len(geo_ensg):>7,}")
print(f"  Fusion genes:           {len(fusion_ensg):>7,}")
print(f"  Mutation genes:         {len(mutation_ensg):>7,}")
print()
print(f"  DepMap ∩ HPA:           {len(depmap_ensg & hpa_ensg):>7,}")
print(f"  DepMap ∩ GEO:           {len(depmap_ensg & geo_ensg):>7,}")
print(f"  HPA ∩ GEO:              {len(hpa_ensg & geo_ensg):>7,}")
print(f"  DepMap ∩ HPA ∩ GEO:     {len(depmap_ensg & hpa_ensg & geo_ensg):>7,}  ← high-confidence set")
print()
print(f"  In DepMap only:         {len(depmap_ensg - hpa_ensg - geo_ensg):>7,}")
print(f"  In HPA only:            {len(hpa_ensg - depmap_ensg - geo_ensg):>7,}")
print(f"  In GEO only:            {len(geo_ensg - depmap_ensg - hpa_ensg):>7,}")
print(f"  Fusions covered by DepMap: {len(fusion_ensg & depmap_ensg):>5,} / {len(fusion_ensg):,}")
print(f"  Mutations covered by DepMap:{len(mutation_ensg & depmap_ensg):>4,} / {len(mutation_ensg):,}")

# ── 3. cell line ID overlap ───────────────────────────────────────────
depmap_ids   = set(sample_info["DepMap_ID"].dropna())

# resolve PR- → ACH- for expression and mutations
rna_pr       = set(depmap_expr.index)
rna_lookup   = depmap_profiles[depmap_profiles["Datatype"]=="rna"].set_index("ProfileID")["ModelID"].to_dict()
expr_ids     = set(filter(None, [rna_lookup.get(p) for p in rna_pr]))

prot_ids     = set(proteomics["Unnamed: 0"].dropna())
sig_ids      = set(signatures["ModelID"].dropna())
metab_ids    = set(metabolomics["DepMap_ID"].dropna())
fusion_ids   = set(fusions["ModelID"].dropna())

mut_pr       = set(mutations["ProfileID"].dropna())
mut_lookup   = depmap_profiles[depmap_profiles["Datatype"].isin(["wes","wgs"])].set_index("ProfileID")["ModelID"].to_dict()
mut_ids      = set(filter(None, [mut_lookup.get(p) for p in mut_pr]))

print()
print("=" * 55)
print("  CELL LINE ID OVERLAP  (each vs Sample Info)")
print("=" * 55)
sources = {
    "DepMap expr (resolved)": expr_ids,
    "Proteomics":             prot_ids,
    "Mutations (resolved)":   mut_ids,
    "Fusions":                fusion_ids,
    "Signatures":             sig_ids,
    "Metabolomics":           metab_ids,
}
for name, ids in sources.items():
    matched   = ids & depmap_ids
    orphaned  = ids - depmap_ids
    pct       = len(matched)/len(ids)*100 if ids else 0
    print(f"  {name:<28} {len(matched):>5,}/{len(ids):,}  ({pct:.0f}%)  orphaned: {len(orphaned):,}")

# ── 4. HPA cell line name matching ────────────────────────────────────
hpa_names        = set(hpa_rna["Cell line"].unique())
sample_names     = set(sample_info["cell_line_name"].dropna())
sample_stripped  = set(sample_info["stripped_cell_line_name"].dropna())
cello_primary    = set(cellosaurus["Identifier (cell line name)"].dropna())
cello_accessions = set(sample_info["RRID"].dropna())
hpa_cvcl         = set(hpa_desc["Cellosaurus ID"].dropna())

exact            = hpa_names & sample_names
via_stripped     = hpa_names & sample_stripped - exact
unmatched        = hpa_names - sample_names - sample_stripped

print()
print("=" * 55)
print("  HPA CELL LINE NAME MATCHING")
print("=" * 55)
print(f"  Unique HPA cell lines:          {len(hpa_names):>5,}")
print(f"  Exact match → cell_line_name:   {len(exact):>5,}")
print(f"  Match via stripped_name:        {len(via_stripped):>5,}")
print(f"  Unmatched (need Cellosaurus):   {len(unmatched):>5,}")
print(f"\n  Sample unmatched names:")
for n in sorted(unmatched)[:10]: print(f"    {n}")

# ── 5. GEO sample → cell line coverage ───────────────────────────────
gsm_cols     = [c for c in geo_expr.columns if c != "Gene"]
geo_info_map = geo_info.set_index("Geo_accession") if "Geo_accession" in geo_info.columns else None

print()
print("=" * 55)
print("  GEO SAMPLE COVERAGE")
print("=" * 55)
print(f"  Total GSM sample columns: {len(gsm_cols):,}")
if geo_info_map is not None:
    mapped      = [g for g in gsm_cols if g in geo_info_map.index]
    has_cvcl    = geo_info[geo_info["Geo_accession"].isin(mapped)]["Cellosaurus_ID"].notna().sum()
    has_name    = geo_info[geo_info["Geo_accession"].isin(mapped)]["cell_line"].notna().sum()
    print(f"  GSMs found in GEO Info:   {len(mapped):,}")
    print(f"  With Cellosaurus_ID:      {has_cvcl:,}  ← direct join to cell_line")
    print(f"  With cell_line name only: {has_name:,}  ← needs name matching")
    print(f"  Unmapped GSMs:            {len(gsm_cols)-len(mapped):,}  ← drop these")
else:
    print("  GEO Info not loaded correctly — fix separator first")

  GENE ID OVERLAP
  DepMap genes:            53,961
  HPA genes:               20,162
  GEO genes:               19,914
  Fusion genes:            25,424
  Mutation genes:          19,798

  DepMap ∩ HPA:            19,896
  DepMap ∩ GEO:            19,894
  HPA ∩ GEO:               16,062
  DepMap ∩ HPA ∩ GEO:      16,060  ← high-confidence set

  In DepMap only:          30,231
  In HPA only:                264
  In GEO only:                 18
  Fusions covered by DepMap: 25,222 / 25,424
  Mutations covered by DepMap:19,635 / 19,798

  CELL LINE ID OVERLAP  (each vs Sample Info)
  DepMap expr (resolved)       1,412/1,479  (95%)  orphaned: 67
  Proteomics                     375/375  (100%)  orphaned: 0
  Mutations (resolved)         1,697/1,744  (97%)  orphaned: 47
  Fusions                      1,428/1,699  (84%)  orphaned: 271
  Signatures                   1,704/1,955  (87%)  orphaned: 251
  Metabolomics                   927/927  (100%)  orphaned: 0

  HPA CELL LINE NAME MATCHIN

In [14]:
import pandas as pd
import re

# ── Master audit function ─────────────────────────────────────────────

def audit_column(series, col_name):
    """Check one column for whitespace, prefix/suffix patterns."""
    results = {"Column": col_name, "Dtype": str(series.dtype)}
    
    if series.dtype != "object":
        results["Leading spaces"]    = "n/a (numeric)"
        results["Trailing spaces"]   = "n/a (numeric)"
        results["Double spaces"]     = "n/a (numeric)"
        results["Has prefix pattern"] = "n/a"
        results["Has suffix pattern"] = "n/a"
        results["Has version suffix"] = "n/a"
        results["Needs stripping"]    = "No"
        results["Sample issues"]      = "—"
        return results
    
    non_null = series.dropna()
    if len(non_null) == 0:
        results["Leading spaces"]    = "ALL NULL"
        results["Trailing spaces"]   = "ALL NULL"
        results["Double spaces"]     = "ALL NULL"
        results["Has prefix pattern"] = "ALL NULL"
        results["Has suffix pattern"] = "ALL NULL"
        results["Has version suffix"] = "ALL NULL"
        results["Needs stripping"]    = "No"
        results["Sample issues"]      = "—"
        return results

    total = len(non_null)
    
    # whitespace checks
    leading  = non_null.str.match(r"^\s").sum()
    trailing = non_null.str.match(r".*\s$").sum()
    internal_double = non_null.str.contains(r"\s{2,}", regex=True).sum()
    
    # prefix patterns (common biological IDs)
    prefixes = {
        "ACH-":  r"^ACH-",
        "PR-":   r"^PR-",
        "ENSG":  r"^ENSG\d",
        "ENST":  r"^ENST\d",
        "ENSP":  r"^ENSP\d",
        "CVCL_": r"^CVCL_",
        "GSM":   r"^GSM\d",
        "GSE":   r"^GSE\d",
        "MC-":   r"^MC-",
        "CDS-":  r"^CDS-",
        "HGNC:": r"^HGNC:",
        "chr":   r"^chr\d",
    }
    
    # suffix patterns
    suffixes = {
        ".version (e.g. .21)": r"\.\d+$",
        "_TISSUE (e.g. _LUNG)": r"_[A-Z]{2,}$",
        "-isoform (e.g. -2)":  r"-\d+$",
        "(brackets)":          r"\(.+\)$",
        "[brackets]":          r"\[.+\]$",
    }
    
    # find matching prefixes
    found_prefixes = []
    for name, pattern in prefixes.items():
        count = non_null.str.contains(pattern, regex=True).sum()
        if count > 0:
            found_prefixes.append(f"{name} ({count:,}/{total:,})")
    
    # find matching suffixes
    found_suffixes = []
    for name, pattern in suffixes.items():
        count = non_null.str.contains(pattern, regex=True).sum()
        if count > 0:
            found_suffixes.append(f"{name} ({count:,}/{total:,})")
    
    # version suffix specifically for ENSG
    version_suffix = non_null.str.contains(r"ENSG\d+\.\d+", regex=True).sum()
    
    results["Leading spaces"]     = f"{leading:,}" if leading > 0 else "None"
    results["Trailing spaces"]    = f"{trailing:,}" if trailing > 0 else "None"
    results["Double spaces"]      = f"{internal_double:,}" if internal_double > 0 else "None"
    results["Has prefix pattern"] = ", ".join(found_prefixes) if found_prefixes else "None"
    results["Has suffix pattern"] = ", ".join(found_suffixes) if found_suffixes else "None"
    results["Has version suffix"] = f"{version_suffix:,}" if version_suffix > 0 else "None"
    results["Needs stripping"]    = "YES" if (leading + trailing + internal_double) > 0 else "No"
    
    # show sample problematic values
    samples = []
    if leading > 0:
        pat_lead = r"^\s"
        val = repr(non_null[non_null.str.match(pat_lead)].iloc[0])
        samples.append("leading: " + val)
    if trailing > 0:
        pat_trail = r".*\s$"
        val = repr(non_null[non_null.str.match(pat_trail)].iloc[0])
        samples.append("trailing: " + val)
    if internal_double > 0:
        pat_double = r"\s{2,}"
        val = repr(non_null[non_null.str.contains(pat_double, regex=True)].iloc[0])
        samples.append("double spaces: " + val)
    results["Sample issues"] = " | ".join(samples) if samples else "—"
    
    return results


def audit_table(df, name, max_cols=None):
    """
    Generate a comprehensive data quality audit report for all columns in a DataFrame.
    
    This function performs detailed column-level audits including whitespace checks,
    identifier prefix/suffix pattern detection, and version suffix identification.
    Results are returned as a color-coded styled DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to audit.
    name : str
        Display name for the dataset (shown in header).
    max_cols : int, optional
        Maximum number of columns to audit (shows first N columns if specified).
    
    Returns
    -------
    pd.io.formats.style.Styler
        Styled DataFrame with audit results including:
            - Column: Column name
            - Dtype: Data type
            - Leading spaces: Count of values with leading whitespace
            - Trailing spaces: Count with trailing whitespace
            - Double spaces: Count with multiple consecutive spaces
            - Has prefix pattern: Detected prefixes (ACH-, PR-, ENSG, CVCL_, GSM, etc.)
            - Has suffix pattern: Detected suffixes (version numbers, _TISSUE, -isoform)
            - Has version suffix: Count of ENSG with version numbers (e.g., .11)
            - Needs stripping: YES if whitespace cleanup needed
            - Sample issues: Example problematic values
    
    Examples
    --------
    >>> audit_table(hpa_rna, "1. HPA RNA")
    >>> audit_table(depmap_expr, "DepMap Expression", max_cols=20)
    
    Notes
    -----
    - Only object (string) columns are analyzed for patterns
    - Numeric columns show "n/a" for pattern checks
    - Index is also audited if it has a name and is object dtype
    - Uses pandas 1.3+ styling API (Styler.map instead of deprecated applymap)
    """
    print(f"\n{'='*65}")
    print(f"  AUDIT: {name}")
    print(f"{'='*65}")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]:,} cols\n")
    
    cols = df.columns.tolist()
    if max_cols and len(cols) > max_cols:
        cols = cols[:max_cols]
        print(f"  (showing first {max_cols} of {df.shape[1]:,} columns)\n")
    
    rows = [audit_column(df[col], col) for col in cols]
    result = pd.DataFrame(rows)
    
    # also check index if it has a name
    if df.index.name and df.index.dtype == "object":
        idx_series = pd.Series(df.index, name=df.index.name)
        idx_result = audit_column(idx_series, f"INDEX: {df.index.name}")
        rows.insert(0, idx_result)
        result = pd.DataFrame(rows)
    
    # colour code the Needs stripping column (updated for pandas 1.3+)
    def highlight(val):
        if val == "YES":
            return "background-color:#f8d7da;color:#721c24"
        elif val == "No":
            return "background-color:#d4edda;color:#155724"
        return ""
    
    # Create styled output with compatibility for different pandas versions
    styled = result.style
    
    # Apply highlighting (pandas 1.3+ uses .map, older uses .applymap)
    if hasattr(styled, 'map'):
        styled = styled.map(highlight, subset=["Needs stripping"])
    else:
        styled = styled.applymap(highlight, subset=["Needs stripping"])
    
    # Hide index (pandas 1.4+ uses .hide(), older uses .hide_index())
    if hasattr(styled, 'hide'):
        styled = styled.hide(axis="index")
    else:
        styled = styled.hide_index()
    
    # Format percentage columns if they exist
    if "Missing %" in result.columns:
        styled = styled.format({"Missing %": "{:.1f}%"})
    
    return styled


# ── Example usage ─────────────────────────────────────────────────────
if __name__ == "__main__":
    # Assuming you have your data loaded
    # audit_table(hpa_rna, "1. HPA RNA")
    # audit_table(depmap_expr, "2. DepMap Expression", max_cols=20)
    # audit_table(mutations, "3. Mutations", max_cols=15)
    pass

In [15]:
audit_table(hpa_rna, "1. HPA RNA")


  AUDIT: 1. HPA RNA
  Shape: 24,315,372 rows × 6 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Gene,object,None,None,None,"ENSG (24,315,372/24,315,372)",None,None,No,—
Gene name,object,None,None,None,"ENSG (455,868/24,315,372), GSE (1,206/24,315,372)","-isoform (e.g. -2) (440,190/24,315,372)",None,No,—
Cell line,object,None,None,None,None,".version (e.g. .21) (221,782/24,315,372), -isoform (e.g. -2) (9,415,654/24,315,372), (brackets) (60,486/24,315,372), [brackets] (262,106/24,315,372)",None,No,—
TPM,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
pTPM,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
nTPM,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [16]:
# DepMap — check index (PR- IDs) + first 15 and last 5 gene columns
depmap_subset = depmap_expr.iloc[:, list(range(15)) + list(range(-5, 0))]
audit_table(depmap_subset, "2. DepMap Expression (first 15 + last 5 cols)")


  AUDIT: 2. DepMap Expression (first 15 + last 5 cols)
  Shape: 1,495 rows × 20 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
INDEX: index,object,None,None,None,"PR- (1,495/1,495)",None,None,No,—
TSPAN6 (ENSG00000000003),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
TNMD (ENSG00000000005),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
DPM1 (ENSG00000000419),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
SCYL3 (ENSG00000000457),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
C1orf112 (ENSG00000000460),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
FGR (ENSG00000000938),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
CFH (ENSG00000000971),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
FUCA2 (ENSG00000001036),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GCLC (ENSG00000001084),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [17]:
# GEO — check Gene col + first 10 GSM cols
geo_subset = geo_expr.iloc[:, :11]
audit_table(geo_subset, "3. GEO Expression (first 11 cols)")


  AUDIT: 3. GEO Expression (first 11 cols)
  Shape: 19,914 rows × 11 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Gene,object,None,None,None,"ENSG (19,914/19,914)",None,None,No,—
GSM101610,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101615,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101616,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101667,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101668,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101671,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101672,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101673,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
GSM101674,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [18]:
# Proteomics — check ID col + first 10 protein cols
prot_subset = proteomics.iloc[:, :11]
audit_table(prot_subset, "4. Proteomics (first 11 cols)")


  AUDIT: 4. Proteomics (first 11 cols)
  Shape: 375 rows × 11 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Unnamed: 0,object,None,None,None,ACH- (375/375),-isoform (e.g. -2) (375/375),None,No,—
A0AV96 (RBM47),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0AVF1 (IFT56),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0AVG3 (TSNARE1),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0AVI4 (TMEM129),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0AVK6 (E2F8),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0AVT1 (UBA6),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0JLT2 (MED19),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0JNW5 (BLTP3B),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
A0MZ66 (SHTN1),float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [19]:
audit_table(fusions, "5. Fusions")


  AUDIT: 5. Fusions
  Shape: 184,237 rows × 30 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Unnamed: 0,int64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
SequencingID,object,None,None,None,"CDS- (184,237/184,237)",None,None,No,—
ModelID,object,None,None,None,"ACH- (184,237/184,237)","-isoform (e.g. -2) (184,237/184,237)",None,No,—
IsDefaultEntryForModel,object,None,None,None,None,None,None,No,—
ModelConditionID,object,None,None,None,"MC- (184,237/184,237)","-isoform (e.g. -2) (39/184,237)",None,No,—
IsDefaultEntryForMC,object,None,None,None,None,None,None,No,—
CanonicalFusionName,object,None,None,None,"ENSG (279/184,237), GSE (40/184,237)",".version (e.g. .21) (33,445/184,237), _TISSUE (e.g. _LUNG) (336/184,237), -isoform (e.g. -2) (407/184,237)",316,No,—
gene1(ENS ID),object,None,None,None,"ENSG (102/184,237), GSE (56/184,237)","(brackets) (184,237/184,237)","170,056",No,—
gene2(ENS ID),object,None,None,None,"ENSG (260/184,237), GSE (15/184,237)","(brackets) (184,237/184,237)","154,675",No,—
TotalReadsInSample,int64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [20]:
# Mutations — only string columns (skip numerics)
mut_str_cols = mutations.select_dtypes(include="object").columns.tolist()
mut_bool_cols = mutations.select_dtypes(include="bool").columns.tolist()
audit_table(mutations[mut_str_cols + mut_bool_cols], "6. Mutations (string + bool cols only)")


  AUDIT: 6. Mutations (string + bool cols only)
  Shape: 1,066,869 rows × 50 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Chrom,object,None,None,None,"chr (1,025,044/1,066,869)",None,None,No,—
Ref,object,None,None,None,None,None,None,No,—
Alt,object,None,None,None,None,None,None,No,—
GT,object,None,None,None,None,None,None,No,—
VariantType,object,None,None,None,None,None,None,No,—
VariantInfo,object,None,None,None,None,None,None,No,—
DNAChange,object,None,None,None,"ENST (1,066,370/1,066,370)",None,None,No,—
ProteinChange,object,None,None,None,None,None,None,No,—
HugoSymbol,object,None,None,None,"GSE (133/1,066,869)",".version (e.g. .21) (1/1,066,869), -isoform (e.g. -2) (5,949/1,066,869)",None,No,—
Exon,object,None,None,None,None,None,None,No,—


In [21]:
audit_table(cellosaurus, "7. Cellosaurus")


  AUDIT: 7. Cellosaurus
  Shape: 152,231 rows × 17 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Identifier (cell line name),object,None,None,None,"ACH- (5/152,230), PR- (3/152,230), GSM (2/152,230), MC- (30/152,230)",".version (e.g. .21) (2,631/152,230), _TISSUE (e.g. _LUNG) (26/152,230), -isoform (e.g. -2) (17,272/152,230), (brackets) (1,505/152,230), [brackets] (1,887/152,230)",None,No,—
Accession (CVCL_xxxx),object,None,None,None,"CVCL_ (152,231/152,231)",None,None,No,—
Secondary accession number(s),object,None,None,None,CVCL_ (542/542),None,None,No,—
Synonyms,object,None,None,None,"PR- (4/72,386), MC- (21/72,386), CDS- (1/72,386)",".version (e.g. .21) (1,261/72,386), _TISSUE (e.g. _LUNG) (44/72,386), -isoform (e.g. -2) (9,325/72,386), (brackets) (955/72,386), [brackets] (14/72,386)",None,No,—
Cross-references,object,None,None,None,None,".version (e.g. .21) (1/150,430), -isoform (e.g. -2) (123/150,430)",None,No,—
References identifiers,object,None,None,None,None,None,None,No,—
Web pages,object,None,None,None,None,".version (e.g. .21) (13/11,630), _TISSUE (e.g. _LUNG) (3/11,630), -isoform (e.g. -2) (784/11,630), (brackets) (3/11,630)",None,No,—
Comments,object,None,None,None,None,None,None,No,—
STR profile data,object,None,None,None,None,".version (e.g. .21) (22/8,734), (brackets) (194/8,734)",None,No,—
Diseases,object,None,None,None,None,".version (e.g. .21) (7/70,990), -isoform (e.g. -2) (20/70,990), (brackets) (119/70,990)",None,No,—


In [22]:
audit_table(depmap_profiles, "8. DepMap Profiles")


  AUDIT: 8. DepMap Profiles
  Shape: 3,830 rows × 5 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
ProfileID,object,None,None,None,"PR- (3,830/3,830)",None,None,No,—
ModelCondition,object,None,None,None,"MC- (3,830/3,830)",None,None,No,—
ModelID,object,None,None,None,"ACH- (3,830/3,830)","-isoform (e.g. -2) (3,830/3,830)",None,No,—
Datatype,object,None,None,None,None,None,None,No,—
WESKit,object,None,None,None,None,None,None,No,—


In [23]:
audit_table(sample_info, "9. Sample Info")


  AUDIT: 9. Sample Info
  Shape: 1,840 rows × 29 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
DepMap_ID,object,None,None,None,"ACH- (1,840/1,840)","-isoform (e.g. -2) (1,840/1,840)",None,No,—
cell_line_name,object,None,None,1,"MC- (3/1,748)",".version (e.g. .21) (15/1,748), _TISSUE (e.g. _LUNG) (2/1,748), -isoform (e.g. -2) (674/1,748), (brackets) (3/1,748), [brackets] (2/1,748)",None,YES,double spaces: 'Hs 343.T'
stripped_cell_line_name,object,None,None,None,None,None,None,No,—
CCLE_Name,object,None,None,None,None,"_TISSUE (e.g. _LUNG) (1,832/1,836)",None,No,—
alias,object,None,None,2,None,".version (e.g. .21) (1/113), -isoform (e.g. -2) (40/113), (brackets) (1/113)",None,YES,"double spaces: 'UPCISCC090, UPCI:SCC090, UPCI-SCC-090'"
COSMICID,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
sex,object,None,None,None,None,None,None,No,—
source,object,None,None,None,None,"(brackets) (1/1,793)",None,No,—
RRID,object,None,None,None,"CVCL_ (1,818/1,818)",None,None,No,—
WTSI_Master_Cell_ID,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [24]:
audit_table(sample_info, "9. Sample Info")


  AUDIT: 9. Sample Info
  Shape: 1,840 rows × 29 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
DepMap_ID,object,None,None,None,"ACH- (1,840/1,840)","-isoform (e.g. -2) (1,840/1,840)",None,No,—
cell_line_name,object,None,None,1,"MC- (3/1,748)",".version (e.g. .21) (15/1,748), _TISSUE (e.g. _LUNG) (2/1,748), -isoform (e.g. -2) (674/1,748), (brackets) (3/1,748), [brackets] (2/1,748)",None,YES,double spaces: 'Hs 343.T'
stripped_cell_line_name,object,None,None,None,None,None,None,No,—
CCLE_Name,object,None,None,None,None,"_TISSUE (e.g. _LUNG) (1,832/1,836)",None,No,—
alias,object,None,None,2,None,".version (e.g. .21) (1/113), -isoform (e.g. -2) (40/113), (brackets) (1/113)",None,YES,"double spaces: 'UPCISCC090, UPCI:SCC090, UPCI-SCC-090'"
COSMICID,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
sex,object,None,None,None,None,None,None,No,—
source,object,None,None,None,None,"(brackets) (1/1,793)",None,No,—
RRID,object,None,None,None,"CVCL_ (1,818/1,818)",None,None,No,—
WTSI_Master_Cell_ID,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [25]:
audit_table(geo_info, "10. GEO Info")


  AUDIT: 10. GEO Info
  Shape: 3,267 rows × 23 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Geo_accession,object,None,None,None,"GSM (3,267/3,267)",None,None,No,—
CEL_file_names,object,None,None,None,"GSM (3,267/3,267)",None,None,No,—
title,object,None,None,4,None,"_TISSUE (e.g. _LUNG) (16/2,329), -isoform (e.g. -2) (202/2,329), (brackets) (16/2,329)",None,YES,"double spaces: 'glioblastoma stem-like cell line, serum free_GS-3 clone 1'"
status,object,None,None,None,None,None,None,No,—
submission_date,object,None,None,None,None,None,None,No,—
last_update_date,object,None,None,None,None,None,None,No,—
type,object,None,None,None,None,None,None,No,—
channel_count,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
source_name_ch1,object,None,None,9,None,"-isoform (e.g. -2) (30/2,329), (brackets) (17/2,329)",None,YES,double spaces: 'HEC-108 Control'
organism_ch1,object,None,None,None,None,None,None,No,—


In [26]:
# Metabolomics — ID cols + first 10 metabolites
metab_subset = metabolomics.iloc[:, :12]
audit_table(metab_subset, "12. Metabolomics (first 12 cols)")


  AUDIT: 12. Metabolomics (first 12 cols)
  Shape: 928 rows × 12 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
CCLE_ID,object,None,None,None,None,_TISSUE (e.g. _LUNG) (928/928),None,No,—
DepMap_ID,object,None,None,None,ACH- (927/927),-isoform (e.g. -2) (927/927),None,No,—
2-aminoadipate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
3-phosphoglycerate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
alpha-glycerophosphate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
4-pyridoxate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
aconitate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
adenine,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
adipate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
alpha-ketoglutarate,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [27]:
# miRNA — ID cols + first 5 cell lines
mirna_subset = mirna.iloc[:, :7]
audit_table(mirna_subset, "13. miRNA (first 7 cols)")


  AUDIT: 13. miRNA (first 7 cols)
  Shape: 734 rows × 7 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Name,object,None,None,None,None,.version (e.g. .21) (734/734),None,No,—
Description,object,None,None,None,None,-isoform (e.g. -2) (395/734),None,No,—
DMS53_LUNG,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
SW1116_LARGE_INTESTINE,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
NCIH1694_LUNG,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
P3HR1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
HUT78_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [28]:
audit_table(signatures, "14. Global Signatures")


  AUDIT: 14. Global Signatures
  Shape: 3,021 rows × 12 cols



Column,Dtype,Leading spaces,Trailing spaces,Double spaces,Has prefix pattern,Has suffix pattern,Has version suffix,Needs stripping,Sample issues
Unnamed: 0,int64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
SequencingID,object,None,None,None,"CDS- (3,021/3,021)",None,None,No,—
ModelID,object,None,None,None,"ACH- (3,021/3,021)","-isoform (e.g. -2) (3,021/3,021)",None,No,—
ModelConditionID,object,None,None,None,"MC- (3,021/3,021)","-isoform (e.g. -2) (2/3,021)",None,No,—
IsDefaultEntryForModel,object,None,None,None,None,None,None,No,—
IsDefaultEntryForMC,object,None,None,None,None,None,None,No,—
MSIScore,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
LoHFraction,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
WGD,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—
CIN,float64,n/a (numeric),n/a (numeric),n/a (numeric),n/a,n/a,n/a,No,—


In [29]:
# ── Summary: every column across all 14 files that needs stripping ────

all_tables = {
    "1. HPA RNA":       hpa_rna,
    "2. DepMap Expr":   depmap_expr.iloc[:, :20],
    "3. GEO Expr":      geo_expr.iloc[:, :15],
    "4. Proteomics":    proteomics.iloc[:, :15],
    "5. Fusions":       fusions,
    "6. Mutations":     mutations.select_dtypes(include="object"),
    "7. Cellosaurus":   cellosaurus,
    "8. Profiles":      depmap_profiles,
    "9. Sample Info":   sample_info,
    "10. GEO Info":     geo_info,
    "11. HPA Desc":     hpa_desc,
    "12. Metabolomics": metabolomics.iloc[:, :15],
    "13. miRNA":        mirna.iloc[:, :10],
    "14. Signatures":   signatures,
}

issues = []
for tbl_name, df in all_tables.items():
    for col in df.columns:
        if df[col].dtype != "object":
            continue
        non_null = df[col].dropna()
        if len(non_null) == 0:
            continue
        leading  = non_null.str.match(r"^\s").sum()
        trailing = non_null.str.match(r".*\s$").sum()
        has_version = non_null.str.contains(r"ENSG\d+\.\d+", regex=True).sum()
        has_brackets = non_null.str.contains(r"\(.+\)$", regex=True).sum()
        has_tissue_suffix = non_null.str.contains(r"_[A-Z]{2,}$", regex=True).sum()
        
        if leading or trailing or has_version or has_brackets or has_tissue_suffix:
            issues.append({
                "Table":          tbl_name,
                "Column":         col,
                "Leading ws":     leading if leading else "",
                "Trailing ws":    trailing if trailing else "",
                "ENSG.version":   has_version if has_version else "",
                "(brackets)":     has_brackets if has_brackets else "",
                "_TISSUE suffix": has_tissue_suffix if has_tissue_suffix else "",
                "Sample":         repr(non_null.iloc[0])[:50]
            })

if issues:
    issue_df = pd.DataFrame(issues)
    def red(val):
        if val and val != "":
            return "background-color:#f8d7da;color:#721c24"
        return ""
    display(issue_df.style.applymap(red, 
        subset=["Leading ws","Trailing ws","ENSG.version","(brackets)","_TISSUE suffix"]
    ).hide(axis="index"))
    print(f"\nTotal columns with issues: {len(issues)}")
else:
    print("No whitespace or suffix issues found across any table!")

AttributeError: 'Styler' object has no attribute 'applymap'

In [ ]:
import re, pandas as pd

"""
Analyze DepMap-specific genes not found in HPA or GEO transcriptomics datasets.

This script investigates the 30,231 genes that appear exclusively in DepMap expression data
(not present in HPA or GEO), characterizing whether they have standard gene symbols
or are bare ENSG identifiers without annotations.

Purpose:
    - Understand what types of genes are unique to DepMap
    - Identify if these represent technical artifacts, non-coding RNAs, 
      or genuine expression features not captured by other platforms
    - Assess data quality and annotation completeness

The script:
    1. Parses DepMap column names to extract ENSG IDs and gene symbols
       Format examples:
         - "TP53 (ENSG00000141510)" → ENSG: ENSG00000141510, Symbol: TP53
         - "ENSG00000141510" → ENSG: ENSG00000141510, Symbol: None (bare)
    
    2. Identifies DepMap-only genes (in DepMap but NOT in HPA and NOT in GEO)
    
    3. Classifies DepMap-only genes into:
       - With gene symbol: Have human-readable names (e.g., TP53, BRCA1)
       - Bare ENSG only: No associated gene symbol (potential non-coding or novel)

Statistics Reported:
    - Total number of DepMap-only genes
    - Count with assignable gene symbols
    - Count with only bare ENSG identifiers
    - Sample of gene symbols (first 15)
    - Sample of bare ENSG IDs (first 10)

Expected Interpretations:
    - High proportion of bare ENSGs may indicate:
        * Non-coding RNAs (lncRNAs, miRNAs, snRNAs)
        * Predicted/novel genes without official symbols
        * Mitochondrial or ribosomal RNA genes
        * Technical artifacts or misannotations
    
    - Genes with symbols but DepMap-only likely represent:
        * Tissue-specific genes not expressed in HPA cell lines
        * Genes discovered after HPA/GEO data collection
        * Cell line-specific expression artifacts

Usage Context:
    This analysis follows gene overlap assessment showing:
        - DepMap: 53,961 genes (most comprehensive)
        - HPA: 20,162 genes
        - GEO: 19,914 genes
    
    The 30,231 DepMap-only genes represent ~56% of DepMap's gene content,
    raising questions about whether these are biological or technical.

Example Decision Points:
    >>> # If most DepMap-only genes are bare ENSGs with no symbols:
    >>> # Consider filtering to only annotated genes for interpretability
    >>> annotated_genes = {e for e in depmap_ensg if depmap_gene_map.get(e)}
    >>> 
    >>> # If many are non-coding RNAs, keep them for specific analyses
    >>> lncRNA_patterns = ['^LINC', '^MIR', '^SNOR']
    >>> potential_noncoding = [sym for sym in have_symbol if any(re.match(p, sym) for p in lncRNA_patterns)]

Notes:
    - The gene symbol extraction regex handles two formats:
        1. "SYMBOL (ENSG00000000000)" - captures both
        2. "ENSG00000000000" - bare ENSG, symbol set to None
    - DepMap column names may contain additional metadata after ENSG
    - Some symbols may be outdated or from previous genome builds
"""

import re, pandas as pd

# ── What are the 30,231 DepMap-only genes? ───────────────────────────

# extract symbol and ensg from depmap columns
depmap_gene_map = {}
for col in depmap_expr.columns:
    m = re.match(r"^(.+?)\s*\((ENSG\d+)", col)
    if m:
        depmap_gene_map[m.group(2)] = m.group(1).strip()  # ensg → symbol
    else:
        m2 = re.search(r"(ENSG\d+)", col)
        if m2:
            depmap_gene_map[m2.group(1)] = None  # bare ensg, no symbol

depmap_only = depmap_ensg - hpa_ensg - geo_ensg

# how many have a symbol vs bare ENSG only
have_symbol  = {e for e in depmap_only if depmap_gene_map.get(e)}
bare_only    = {e for e in depmap_only if not depmap_gene_map.get(e)}

print(f"DepMap-only genes:           {len(depmap_only):,}")
print(f"  With gene symbol:          {len(have_symbol):,}")
print(f"  Bare ENSG only (no symbol):{len(bare_only):,}")
print(f"\nSample with symbols:  {[depmap_gene_map[e] for e in list(have_symbol)[:15]]}")
print(f"\nSample bare ENSGs:    {list(bare_only)[:10]}")

# ── What are the 30,231 DepMap-only genes? ───────────────────────────

# extract symbol and ensg from depmap columns
depmap_gene_map = {}
for col in depmap_expr.columns:
    m = re.match(r"^(.+?)\s*\((ENSG\d+)", col)
    if m:
        depmap_gene_map[m.group(2)] = m.group(1).strip()  # ensg → symbol
    else:
        m2 = re.search(r"(ENSG\d+)", col)
        if m2:
            depmap_gene_map[m2.group(1)] = None  # bare ensg, no symbol

depmap_only = depmap_ensg - hpa_ensg - geo_ensg

# how many have a symbol vs bare ENSG only
have_symbol  = {e for e in depmap_only if depmap_gene_map.get(e)}
bare_only    = {e for e in depmap_only if not depmap_gene_map.get(e)}

print(f"DepMap-only genes:           {len(depmap_only):,}")
print(f"  With gene symbol:          {len(have_symbol):,}")
print(f"  Bare ENSG only (no symbol):{len(bare_only):,}")
print(f"\nSample with symbols:  {[depmap_gene_map[e] for e in list(have_symbol)[:15]]}")
print(f"\nSample bare ENSGs:    {list(bare_only)[:10]}")

DepMap-only genes:           30,231
  With gene symbol:          29,866
  Bare ENSG only (no symbol):365

Sample with symbols:  ['AL356737.1', 'AC104088.3', 'CCT6P4', 'AL354980.2', 'RAC1P6', 'AC091132.2', 'AC116562.3', 'AC019211.1', 'AL449403.3', 'AC004801.5', 'LINC01209', 'AC069549.1', 'AC009229.2', 'MYLK-AS2', 'BX664727.1']

Sample bare ENSGs:    ['ENSG00000273524', 'ENSG00000288685', 'ENSG00000278109', 'ENSG00000278554', 'ENSG00000274357', 'ENSG00000288725', 'ENSG00000252744', 'ENSG00000201398', 'ENSG00000252305', 'ENSG00000274701']


In [30]:
# ── Check HPA column format more carefully ───────────────────────────
# HPA uses ENSG with version suffixes? check
print("HPA Gene column samples:")
print(hpa_rna["Gene"].head(10).tolist())

# do any HPA ENSGs have version suffixes like .1 .2?
hpa_has_version = hpa_rna["Gene"].str.contains(r"\.\d+$", regex=True)
print(f"\nHPA genes with version suffix: {hpa_has_version.sum():,}")

# do any GEO ENSGs have version suffixes?
geo_has_version = geo_expr["Gene"].str.contains(r"\.\d+$", regex=True)
print(f"GEO genes with version suffix:  {geo_has_version.sum():,}")

# DepMap: are the bare ENSG columns (no symbol) versioned?
bare_sample = list(bare_only)[:20]
print(f"\nDepMap bare ENSG sample: {bare_sample}")

HPA Gene column samples:
['ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003', 'ENSG00000000003']

HPA genes with version suffix: 0
GEO genes with version suffix:  0

DepMap bare ENSG sample: ['ENSG00000273524', 'ENSG00000288685', 'ENSG00000278109', 'ENSG00000278554', 'ENSG00000274357', 'ENSG00000288725', 'ENSG00000252744', 'ENSG00000201398', 'ENSG00000252305', 'ENSG00000274701', 'ENSG00000274206', 'ENSG00000221633', 'ENSG00000278374', 'ENSG00000237372', 'ENSG00000221332', 'ENSG00000288714', 'ENSG00000275631', 'ENSG00000268154', 'ENSG00000207034', 'ENSG00000269103']


In [ ]:
"""
Investigate ENSG identifier formatting discrepancies across multi-omics datasets.

This script examines whether ENSG (Ensembl Gene ID) identifiers across different 
datasets (HPA, GEO, DepMap) contain version suffixes (e.g., ENSG00000141510.11)
or use bare identifiers (ENSG00000141510). Version suffixes can cause mismatches
in gene overlap analysis if not properly stripped.

Purpose:
    - Detect version suffixes in HPA and GEO ENSG identifiers
    - Compare formatting conventions across platforms
    - Identify potential join issues caused by versioning
    - Examine DepMap bare ENSG columns for version patterns

Background:
    Ensembl GENCODE releases assign version numbers to gene IDs (e.g., .11)
    Different datasets may use:
        - Bare ENSG (no version): ENSG00000141510
        - Versioned ENSG: ENSG00000141510.11
        - Symbol + version: TP53 (ENSG00000141510.11)
    
    Version mismatches cause false negatives in set operations.

Script Components:
    1. HPA Format Check:
        - Display sample of Gene column values
        - Count genes with version suffixes (pattern: .digits at end)
    
    2. GEO Format Check:
        - Count genes with version suffixes
    
    3. DepMap Bare ENSG Sample:
        - Examine 20 bare ENSG identifiers for version patterns
        - Determine if version stripping is needed

Expected Output Interpretation:
    - If hpa_has_version.sum() > 0: HPA uses versioned ENSGs
    - If geo_has_version.sum() > 0: GEO uses versioned ENSGs  
    - If DepMap bare ENSGs contain versions: Stripping needed before comparison
    
Implications for Analysis:
    - Earlier overlap counts (16,060 triple intersection) may be underestimates
    - Version suffixes should be stripped for accurate ID matching
    - Stripping .\d+$ pattern: ENSG00000141510.11 → ENSG00000141510

Recommended Action:
    >>> # Strip versions from all datasets before overlap analysis
    >>> hpa_rna['Gene_stripped'] = hpa_rna['Gene'].str.replace(r'\.\d+$', '', regex=True)
    >>> geo_expr['Gene_stripped'] = geo_expr['Gene'].str.replace(r'\.\d+$', '', regex=True)
    >>> # Recalculate overlaps with stripped IDs

Example Workflow:
    >>> # Original (may miss matches)
    >>> 'ENSG00000141510' in set(hpa_rna['Gene'])  # False if HPA has version
    >>> # After stripping
    >>> hpa_stripped = hpa_rna['Gene'].str.replace(r'\.\d+$', '', regex=True)
    >>> 'ENSG00000141510' in set(hpa_stripped)    # True
"""

# ── Check HPA column format more carefully ───────────────────────────
# HPA uses ENSG with version suffixes? check
print("HPA Gene column samples:")
print(hpa_rna["Gene"].head(10).tolist())

# do any HPA ENSGs have version suffixes like .1 .2?
hpa_has_version = hpa_rna["Gene"].str.contains(r"\.\d+$", regex=True)
print(f"\nHPA genes with version suffix: {hpa_has_version.sum():,}")

# do any GEO ENSGs have version suffixes?
geo_has_version = geo_expr["Gene"].str.contains(r"\.\d+$", regex=True)
print(f"GEO genes with version suffix:  {geo_has_version.sum():,}")

# DepMap: are the bare ENSG columns (no symbol) versioned?
bare_sample = list(bare_only)[:20]
print(f"\nDepMap bare ENSG sample: {bare_sample}")

=== After stripping version suffixes ===
DepMap ∩ HPA:        19,896  (was 19,896)
DepMap ∩ GEO:        19,894  (was 19,894)
All 3 RNA:           16,060  (was 16,060)

DepMap-only after strip: 30,231  (was 30,231)


In [32]:
# ── Check symbol synonyms ─────────────────────────────────────────────
# maybe HPA uses a different symbol that maps to same ENSG
# compare HPA Gene name column vs DepMap symbols

hpa_symbols = set(hpa_rna["Gene name"].unique())
depmap_symbols = set(v for v in depmap_gene_map.values() if v)

only_in_depmap_sym  = depmap_symbols - hpa_symbols
only_in_hpa_sym     = hpa_symbols - depmap_symbols
in_both_sym         = depmap_symbols & hpa_symbols

print("=== Gene symbol comparison (DepMap vs HPA) ===")
print(f"DepMap symbols:          {len(depmap_symbols):,}")
print(f"HPA symbols:             {len(hpa_symbols):,}")
print(f"Shared symbols:          {len(in_both_sym):,}")
print(f"DepMap-only symbols:     {len(only_in_depmap_sym):,}")
print(f"HPA-only symbols:        {len(only_in_hpa_sym):,}")
print(f"\nSample DepMap-only symbols: {list(only_in_depmap_sym)[:15]}")
print(f"Sample HPA-only symbols:    {list(only_in_hpa_sym)[:15]}")

=== Gene symbol comparison (DepMap vs HPA) ===
DepMap symbols:          53,529
HPA symbols:             20,151
Shared symbols:          19,417
DepMap-only symbols:     34,112
HPA-only symbols:        734

Sample DepMap-only symbols: ['LINC00457', 'AC011593.1', 'AP004833.3', 'RNU1-105P', 'BNIP3P7', 'AL807776.1', 'AC019118.1', 'RPL35AP9', 'SNORA7A', 'AC114760.1', 'FCF1P2', 'CCT5P2', 'AC006927.5', 'LINC01088', 'ATP5MC1P1']
Sample HPA-only symbols:    ['ENSG00000173366', 'IGHD6-19', 'METTL13', 'IGHD4-4', 'ENSG00000284732', 'AKR1C8', 'LDAF1', 'TRAJ44', 'IGHD4-23', 'TRAJ23', 'ENSG00000273217', 'POLGARF', 'ENSG00000249141', 'ENSG00000258881', 'ENSG00000288671']


In [35]:
# ── Biotype check — what are those 30k DepMap-only genes? ────────────
# use mygene to look up a sample of 50 DepMap-only genes
# and check their biotype (protein_coding vs lncRNA vs pseudogene etc)
# install if needed: pip install mygene

try:
    import mygene
    mg = mygene.MyGeneInfo()

    sample_ensg = list(have_symbol)[:100]
    sample_syms = [depmap_gene_map[e] for e in sample_ensg]

    result = mg.querymany(sample_syms,
                          scopes="symbol",
                          fields="ensembl.gene,type_of_gene",
                          species="human",
                          as_dataframe=True)

    print("Gene biotype breakdown for 100 DepMap-only genes:")
    print(result["type_of_gene"].value_counts())

except ImportError:
    print("mygene not installed — run: pip install mygene")
    print("In the meantime, inspect symbol names manually:")
    sample = list(have_symbol)[:30]
    syms   = [depmap_gene_map[e] for e in sample]
    print("Sample symbols:", syms)
    print("\nNames starting with LINC (long non-coding):", 
          [s for s in syms if str(s).startswith("LINC")])
    print("Names starting with MIR (microRNA):", 
          [s for s in syms if str(s).startswith("MIR")])
    print("Names ending in -AS (antisense):", 
          [s for s in syms if str(s).endswith("-AS1") or str(s).endswith("-AS2")])

13 input query terms found dup hits:	[('CCT6P4', 2), ('RAC1P6', 2), ('CASP1P2', 2), ('FTH1P10', 2), ('RPS3AP27', 2), ('RFKP2', 2), ('IGKV
60 input query terms found no hit:	['AL356737.1', 'AC104088.3', 'AL354980.2', 'AC091132.2', 'AC116562.3', 'AC019211.1', 'AL449403.3', '


Gene biotype breakdown for 100 DepMap-only genes:
type_of_gene
pseudo    28
ncRNA     11
Name: count, dtype: int64


In [36]:
import re

depmap_only_symbols = [depmap_gene_map[e] for e in depmap_only if depmap_gene_map.get(e)]

def classify(sym):
    if not sym: return "bare_ensg"
    s = str(sym)
    if re.match(r"^LINC\d",       s): return "lncRNA (LINC)"
    if re.match(r"^SNOR[A-Z]",    s): return "snoRNA"
    if re.match(r"^MIR\d",        s): return "miRNA"
    if re.match(r"^RNU\d",        s): return "snRNA"
    if re.match(r"^RN[0-9]",      s): return "ncRNA (RN)"
    if re.search(r"-AS\d?$",       s): return "antisense RNA"
    if re.search(r"P\d+$",         s): return "pseudogene"
    if re.match(r"^(AC|AL|AP|BX|AF|AJ|AY|CR|FJ|Z)\d{4,}", s): return "uncharacterised locus"
    if re.match(r"^FAM\d",         s): return "uncharacterised family"
    if re.match(r"^RP\d+-",        s): return "uncharacterised locus"
    return "other / check manually"

from collections import Counter
counts = Counter(classify(s) for s in depmap_only_symbols)
counts["bare ENSG (no symbol)"] = len(bare_only)

print("DepMap-only gene breakdown:")
total = 0
for cat, n in sorted(counts.items(), key=lambda x: -x[1]):
    pct = n / len(depmap_only) * 100
    print(f"  {cat:<35} {n:>6,}  ({pct:.1f}%)")
    total += n
print(f"  {'TOTAL':<35} {total:>6,}")

# also check HPA-only symbols — they look like newer gene names
print("\nHPA-only symbols sample (these may be renamed genes):")
hpa_only_symbols = hpa_symbols - depmap_symbols
print(f"  Total HPA-only symbols: {len(hpa_only_symbols):,}")
# how many look like proper gene names vs ENSG placeholders
proper = [s for s in hpa_only_symbols if not str(s).startswith("ENSG")]
ensg_placeholders = [s for s in hpa_only_symbols if str(s).startswith("ENSG")]
print(f"  Proper gene symbols:    {len(proper):,}  — {proper[:10]}")
print(f"  ENSG placeholders:      {len(ensg_placeholders):,}  — {ensg_placeholders[:5]}")

DepMap-only gene breakdown:
  uncharacterised locus               17,189  (56.9%)
  pseudogene                           6,725  (22.2%)
  other / check manually               2,260  (7.5%)
  lncRNA (LINC)                        1,356  (4.5%)
  ncRNA (RN)                             931  (3.1%)
  antisense RNA                          831  (2.7%)
  bare ENSG (no symbol)                  365  (1.2%)
  snRNA                                  222  (0.7%)
  snoRNA                                 160  (0.5%)
  uncharacterised family                 133  (0.4%)
  miRNA                                   59  (0.2%)
  TOTAL                               30,231

HPA-only symbols sample (these may be renamed genes):
  Total HPA-only symbols: 734
  Proper gene symbols:    356  — ['IGHD6-19', 'METTL13', 'IGHD4-4', 'AKR1C8', 'LDAF1', 'TRAJ44', 'IGHD4-23', 'TRAJ23', 'POLGARF', 'TRBJ1-2']
  ENSG placeholders:      378  — ['ENSG00000173366', 'ENSG00000284732', 'ENSG00000273217', 'ENSG00000249141', 'ENSG0

In [40]:
# ── Step 2: Direct pairwise connections ───────────────────────────────

print("="*65)
print("  PAIRWISE CONNECTIONS — can file X join to file Y directly?")
print("="*65)

# --- 1. HPA RNA connections ---
print("\n[1. HPA RNA] connects via:")

# → HPA Desc (same cell line names?)
overlap = set(hpa_rna["Cell line"].unique()) & set(hpa_desc["Cell line"].unique())
print(f"  → 11. HPA Desc       via 'Cell line' name         {len(overlap):,} matches")

# → Sample Info (cell line name)
overlap = set(hpa_rna["Cell line"].unique()) & set(sample_info["cell_line_name"].dropna())
print(f"  → 9.  Sample Info    via name exact match         {len(overlap):,} matches")

overlap2 = set(hpa_rna["Cell line"].unique()) & set(sample_info["stripped_cell_line_name"].dropna())
print(f"  → 9.  Sample Info    via stripped name match      {len(overlap2):,} matches")

# → DepMap Expr (shared Ensembl gene IDs in columns?)
hpa_genes = set(hpa_rna["Gene"].unique())
depmap_genes = set()
for c in depmap_expr.columns:
    m = re.search(r"(ENSG\d+)", c)
    if m: depmap_genes.add(m.group(1))
overlap = hpa_genes & depmap_genes
print(f"  → 2.  DepMap Expr    via Ensembl gene ID          {len(overlap):,} genes shared")

# → GEO Expr
geo_genes = set(geo_expr["Gene"].unique())
overlap = hpa_genes & geo_genes
print(f"  → 3.  GEO Expr      via Ensembl gene ID          {len(overlap):,} genes shared")

# → Cellosaurus (via HPA Desc Cellosaurus ID)
hpa_cvcl = set(hpa_desc["Cellosaurus ID"].dropna())
cello_cvcl = set(cellosaurus["Accession (CVCL_xxxx)"].dropna())
overlap = hpa_cvcl & cello_cvcl
print(f"  → 7.  Cellosaurus    via HPA Desc CVCL ID         {len(overlap):,} matches")


# --- 2. DepMap Expr connections ---
print("\n[2. DepMap Expr] connects via:")

# → Profiles (PR- index)
expr_prs = set(depmap_expr.index)
prof_prs = set(depmap_profiles["ProfileID"].dropna())
overlap = expr_prs & prof_prs
print(f"  → 8.  Profiles       via PR- ProfileID            {len(overlap):,} matches")

# → Sample Info (via Profiles → ACH-)
rna_profiles = depmap_profiles[depmap_profiles["Datatype"]=="rna"]
rna_lookup = rna_profiles.set_index("ProfileID")["ModelID"].to_dict()
resolved = set(filter(None, [rna_lookup.get(p) for p in expr_prs]))
overlap = resolved & set(sample_info["DepMap_ID"].dropna())
print(f"  → 9.  Sample Info    via PR- → Profiles → ACH-   {len(overlap):,} matches")

# → HPA RNA (shared genes)
print(f"  → 1.  HPA RNA        via Ensembl gene ID          {len(depmap_genes & hpa_genes):,} genes shared")

# → GEO Expr (shared genes)
print(f"  → 3.  GEO Expr      via Ensembl gene ID          {len(depmap_genes & geo_genes):,} genes shared")

# → Mutations (both use PR- → resolved via Profiles)
mut_prs = set(mutations["ProfileID"].dropna().unique())
shared_prs = expr_prs & mut_prs
print(f"  → 6.  Mutations      via shared PR- IDs           {len(shared_prs):,} profiles")


# --- 3. GEO Expr connections ---
print("\n[3. GEO Expr] connects via:")

gsm_cols = [c for c in geo_expr.columns if c != "Gene"]

# → GEO Info
if geo_info.shape[1] > 1:
    geo_gsms = set(geo_info["Geo_accession"].dropna()) if "Geo_accession" in geo_info.columns else set()
    overlap = set(gsm_cols) & geo_gsms
    print(f"  → 10. GEO Info       via GSM accession            {len(overlap):,} matches")

    # → Cellosaurus (via GEO Info Cellosaurus_ID)
    geo_cvcl = set(geo_info["Cellosaurus_ID"].dropna()) if "Cellosaurus_ID" in geo_info.columns else set()
    overlap2 = geo_cvcl & cello_cvcl
    print(f"  → 7.  Cellosaurus    via GEO Info CVCL ID         {len(overlap2):,} matches")

    # → Sample Info (via GEO Info cell_line name)
    if "cell_line" in geo_info.columns:
        geo_names = set(geo_info["cell_line"].dropna())
        overlap3 = geo_names & set(sample_info["cell_line_name"].dropna())
        print(f"  → 9.  Sample Info    via GEO Info cell_line name  {len(overlap3):,} matches")
else:
    print(f"  → 10. GEO Info       BROKEN (1 column — fix separator)")

# → HPA RNA / DepMap Expr (shared genes)
print(f"  → 1.  HPA RNA        via Ensembl gene ID          {len(geo_genes & hpa_genes):,} genes shared")
print(f"  → 2.  DepMap Expr    via Ensembl gene ID          {len(geo_genes & depmap_genes):,} genes shared")


# --- 4. Proteomics connections ---
print("\n[4. Proteomics] connects via:")

prot_ids = set(proteomics["Unnamed: 0"].dropna())
overlap = prot_ids & set(sample_info["DepMap_ID"].dropna())
print(f"  → 9.  Sample Info    via ACH- model ID            {len(overlap):,} matches")

# extract gene symbols from proteomics columns
prot_symbols = set()
for c in proteomics.columns:
    m = re.search(r"\((\w+)\)$", c)
    if m: prot_symbols.add(m.group(1))

# → DepMap Expr (shared gene symbols)
depmap_symbols = set()
for c in depmap_expr.columns:
    m = re.match(r"^(.+?)\s*\(", c)
    if m: depmap_symbols.add(m.group(1).strip())

overlap = prot_symbols & depmap_symbols
print(f"  → 2.  DepMap Expr    via gene symbol               {len(overlap):,} genes shared")

# → HPA RNA (symbol match)
hpa_symbols = set(hpa_rna["Gene name"].unique())
overlap = prot_symbols & hpa_symbols
print(f"  → 1.  HPA RNA        via gene symbol               {len(overlap):,} genes shared")

# → Mutations (shared ACH- cell lines)
overlap = prot_ids & set(filter(None, [mut_lookup.get(p) for p in mut_prs]))
print(f"  → 6.  Mutations      via shared ACH- IDs           {len(overlap):,} cell lines")

# → Metabolomics (shared ACH-)
overlap = prot_ids & set(metabolomics["DepMap_ID"].dropna())
print(f"  → 12. Metabolomics   via ACH- model ID            {len(overlap):,} cell lines")


# --- 5. Fusions connections ---
print("\n[5. Fusions] connects via:")
fusion_ids = set(fusions["ModelID"].dropna().unique())
overlap = fusion_ids & set(sample_info["DepMap_ID"].dropna())
print(f"  → 9.  Sample Info    via ACH- ModelID              {len(overlap):,} matches")

# gene overlap with expression
def strip_ver(val):
    m = re.search(r"(ENSG\d+)", str(val)) if pd.notna(val) else None
    return m.group(1) if m else None

fusion_genes = set(filter(None,
    fusions["gene1(ENS ID)"].map(strip_ver).tolist() +
    fusions["gene2(ENS ID)"].map(strip_ver).tolist()))
print(f"  → 2.  DepMap Expr    via Ensembl gene ID          {len(fusion_genes & depmap_genes):,} genes shared")
print(f"  → 1.  HPA RNA        via Ensembl gene ID          {len(fusion_genes & hpa_genes):,} genes shared")

# → Mutations (shared cell lines)
overlap = fusion_ids & set(filter(None, [mut_lookup.get(p) for p in mut_prs]))
print(f"  → 6.  Mutations      via shared ACH- IDs           {len(overlap):,} cell lines")


# --- 6. Mutations connections ---
print("\n[6. Mutations] connects via:")
mut_resolved = set(filter(None, [mut_lookup.get(p) for p in mut_prs]))
print(f"  → 8.  Profiles       via PR- ProfileID            {len(mut_prs & prof_prs):,} matches")
print(f"  → 9.  Sample Info    via PR- → Profiles → ACH-   {len(mut_resolved & set(sample_info['DepMap_ID'].dropna())):,} matches")

mutation_genes = set(mutations["EnsemblGeneID"].dropna().unique())
print(f"  → 2.  DepMap Expr    via Ensembl gene ID          {len(mutation_genes & depmap_genes):,} genes shared")
print(f"  → 1.  HPA RNA        via Ensembl gene ID          {len(mutation_genes & hpa_genes):,} genes shared")
print(f"  → 5.  Fusions        via shared ACH- IDs           {len(mut_resolved & fusion_ids):,} cell lines")


# --- 7. Cellosaurus connections ---
print("\n[7. Cellosaurus] connects via:")

cello_names = set(cellosaurus["Identifier (cell line name)"].dropna())
rrid_set = set(sample_info["RRID"].dropna())
overlap = rrid_set & cello_cvcl
print(f"  → 9.  Sample Info    via RRID = CVCL accession    {len(overlap):,} matches")
print(f"  → 11. HPA Desc       via CVCL accession           {len(hpa_cvcl & cello_cvcl):,} matches")

if geo_info.shape[1] > 1 and "Cellosaurus_ID" in geo_info.columns:
    geo_cvcl = set(geo_info["Cellosaurus_ID"].dropna())
    print(f"  → 10. GEO Info       via CVCL accession           {len(geo_cvcl & cello_cvcl):,} matches")

# name matching
print(f"  → 1.  HPA RNA        via cell line name            {len(set(hpa_rna['Cell line'].unique()) & cello_names):,} matches")
print(f"  → 9.  Sample Info    via cell_line_name            {len(set(sample_info['cell_line_name'].dropna()) & cello_names):,} matches")


# --- 8. Profiles connections ---
print("\n[8. Profiles] connects via:")
print(f"  → 2.  DepMap Expr    via PR- ProfileID            {len(expr_prs & prof_prs):,} matches")
print(f"  → 6.  Mutations      via PR- ProfileID            {len(mut_prs & prof_prs):,} matches")
print(f"  → 9.  Sample Info    via ACH- ModelID              {len(set(depmap_profiles['ModelID'].dropna()) & set(sample_info['DepMap_ID'].dropna())):,} matches")


# --- 9. Sample Info connections ---
print("\n[9. Sample Info] connects via:")
print(f"  → 7.  Cellosaurus    via RRID = CVCL              {len(rrid_set & cello_cvcl):,} matches")
print(f"  → 2.  DepMap Expr    via PR- (through Profiles)   {len(resolved & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 4.  Proteomics     via ACH-                     {len(prot_ids & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 5.  Fusions        via ACH-                     {len(fusion_ids & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 12. Metabolomics   via ACH-                     {len(set(metabolomics['DepMap_ID'].dropna()) & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 14. Signatures     via ACH-                     {len(set(signatures['ModelID'].dropna()) & set(sample_info['DepMap_ID'].dropna())):,}")

# miRNA → Sample Info via CCLE_Name
mirna_cols_set = set(c for c in mirna.columns if c not in ["Name","Description"])
ccle_names_set = set(sample_info["CCLE_Name"].dropna())
print(f"  → 13. miRNA          via CCLE_Name columns        {len(mirna_cols_set & ccle_names_set):,}")


# --- 10. GEO Info connections ---
print("\n[10. GEO Info] connects via:")
if geo_info.shape[1] > 1:
    print(f"  → 3.  GEO Expr      via GSM accession            {len(set(gsm_cols) & set(geo_info['Geo_accession'].dropna())):,}")
    if "Cellosaurus_ID" in geo_info.columns:
        print(f"  → 7.  Cellosaurus    via CVCL ID                 {len(set(geo_info['Cellosaurus_ID'].dropna()) & cello_cvcl):,}")
    if "cell_line" in geo_info.columns:
        print(f"  → 9.  Sample Info    via cell_line name           {len(set(geo_info['cell_line'].dropna()) & set(sample_info['cell_line_name'].dropna())):,}")
        print(f"  → 1.  HPA RNA        via cell_line name           {len(set(geo_info['cell_line'].dropna()) & set(hpa_rna['Cell line'].unique())):,}")
else:
    print(f"  FIX GEO INFO SEPARATOR FIRST")


# --- 11. HPA Desc connections ---
print("\n[11. HPA Desc] connects via:")
print(f"  → 1.  HPA RNA        via cell line name            {len(set(hpa_desc['Cell line'].unique()) & set(hpa_rna['Cell line'].unique())):,}")
print(f"  → 7.  Cellosaurus    via CVCL accession           {len(hpa_cvcl & cello_cvcl):,}")
print(f"  → 9.  Sample Info    via cell_line_name            {len(set(hpa_desc['Cell line'].unique()) & set(sample_info['cell_line_name'].dropna())):,}")
print(f"  → 9.  Sample Info    via RRID=CVCL                {len(hpa_cvcl & rrid_set):,}")


# --- 12. Metabolomics connections ---
print("\n[12. Metabolomics] connects via:")
print(f"  → 9.  Sample Info    via ACH- DepMap_ID           {len(set(metabolomics['DepMap_ID'].dropna()) & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 4.  Proteomics     via ACH-                     {len(set(metabolomics['DepMap_ID'].dropna()) & prot_ids):,}")
print(f"  → 13. miRNA          via CCLE_ID = col names      {len(set(metabolomics['CCLE_ID'].dropna()) & mirna_cols_set):,}")
print(f"  → 5.  Fusions        via ACH-                     {len(set(metabolomics['DepMap_ID'].dropna()) & fusion_ids):,}")
print(f"  → 14. Signatures     via ACH-                     {len(set(metabolomics['DepMap_ID'].dropna()) & set(signatures['ModelID'].dropna())):,}")


# --- 13. miRNA connections ---
print("\n[13. miRNA] connects via:")
print(f"  → 9.  Sample Info    via CCLE_Name columns        {len(mirna_cols_set & ccle_names_set):,}")
print(f"  → 12. Metabolomics   via CCLE_ID = col names      {len(mirna_cols_set & set(metabolomics['CCLE_ID'].dropna())):,}")

# try stripped match
mirna_stripped = set(c.split("_")[0] for c in mirna_cols_set)
si_stripped = set(sample_info["stripped_cell_line_name"].dropna())
print(f"  → 9.  Sample Info    via stripped col name         {len(mirna_stripped & si_stripped):,}")


# --- 14. Signatures connections ---
print("\n[14. Signatures] connects via:")
print(f"  → 9.  Sample Info    via ACH- ModelID              {len(set(signatures['ModelID'].dropna()) & set(sample_info['DepMap_ID'].dropna())):,}")
print(f"  → 4.  Proteomics     via ACH-                     {len(set(signatures['ModelID'].dropna()) & prot_ids):,}")
print(f"  → 5.  Fusions        via ACH-                     {len(set(signatures['ModelID'].dropna()) & fusion_ids):,}")
print(f"  → 12. Metabolomics   via ACH-                     {len(set(signatures['ModelID'].dropna()) & set(metabolomics['DepMap_ID'].dropna())):,}")

  PAIRWISE CONNECTIONS — can file X join to file Y directly?

[1. HPA RNA] connects via:
  → 11. HPA Desc       via 'Cell line' name         1,206 matches
  → 9.  Sample Info    via name exact match         862 matches
  → 9.  Sample Info    via stripped name match      209 matches
  → 2.  DepMap Expr    via Ensembl gene ID          19,896 genes shared
  → 3.  GEO Expr      via Ensembl gene ID          16,062 genes shared
  → 7.  Cellosaurus    via HPA Desc CVCL ID         1,197 matches

[2. DepMap Expr] connects via:
  → 8.  Profiles       via PR- ProfileID            1,495 matches
  → 9.  Sample Info    via PR- → Profiles → ACH-   1,412 matches
  → 1.  HPA RNA        via Ensembl gene ID          19,896 genes shared
  → 3.  GEO Expr      via Ensembl gene ID          19,894 genes shared
  → 6.  Mutations      via shared PR- IDs           0 profiles

[3. GEO Expr] connects via:
  → 10. GEO Info       via GSM accession            3,267 matches
  → 7.  Cellosaurus    via GEO Info CVCL ID 

In [41]:
# Confirm expression and mutations connect via ACH-
expr_achs = set(filter(None, [rna_lookup.get(p) for p in depmap_expr.index]))
mut_achs  = set(filter(None, [mut_lookup.get(p) for p in mutations["ProfileID"].dropna().unique()]))

shared_achs = expr_achs & mut_achs
print(f"Expression cell lines (ACH): {len(expr_achs):,}")
print(f"Mutation cell lines (ACH):   {len(mut_achs):,}")
print(f"Shared via ACH (indirect):   {len(shared_achs):,}")

Expression cell lines (ACH): 1,479
Mutation cell lines (ACH):   1,744
Shared via ACH (indirect):   1,407


In [42]:
# Why only 797 GEO CVCL matches against 152k Cellosaurus entries?
geo_cvcl_all = set(geo_info["Cellosaurus_ID"].dropna())
cello_all    = set(cellosaurus["Accession (CVCL_xxxx)"].dropna())

print(f"Unique CVCL IDs in GEO Info:     {len(geo_cvcl_all):,}")
print(f"Unique CVCL IDs in Cellosaurus:  {len(cello_all):,}")
print(f"Matched:                         {len(geo_cvcl_all & cello_all):,}")
print(f"In GEO but NOT in Cellosaurus:   {len(geo_cvcl_all - cello_all):,}")

# sample the unmatched
unmatched_geo_cvcl = geo_cvcl_all - cello_all
print(f"\nSample unmatched GEO CVCL IDs: {list(unmatched_geo_cvcl)[:10]}")

# also: why only 197 GEO cell_line names match Sample Info?
geo_cell_names = set(geo_info["cell_line"].dropna())
si_names       = set(sample_info["cell_line_name"].dropna())
si_stripped    = set(sample_info["stripped_cell_line_name"].dropna())

print(f"\nGEO unique cell_line names:     {len(geo_cell_names):,}")
print(f"Match vs cell_line_name:        {len(geo_cell_names & si_names):,}")
print(f"Match vs stripped_name:         {len(geo_cell_names & si_stripped):,}")

# sample unmatched
unmatched_geo_names = geo_cell_names - si_names - si_stripped
print(f"Unmatched GEO names:            {len(unmatched_geo_names):,}")
print(f"Sample: {sorted(unmatched_geo_names)[:10]}")

Unique CVCL IDs in GEO Info:     797
Unique CVCL IDs in Cellosaurus:  152,231
Matched:                         797
In GEO but NOT in Cellosaurus:   0

Sample unmatched GEO CVCL IDs: []

GEO unique cell_line names:     1,004
Match vs cell_line_name:        197
Match vs stripped_name:         470
Unmatched GEO names:            449
Sample: [' GS-3', ' GS-3-2', ' GS-5', ' GS-5-2', ' GS-8', ' GS-8-2', ' GS-9', ' GS-9-2', '1A6', '537MEL ']


In [43]:
# How many of the 281 name-unmatched HPA lines resolve via CVCL?
hpa_names_all    = set(hpa_rna["Cell line"].unique())
matched_by_name  = set(sample_info["cell_line_name"].dropna()) | set(sample_info["stripped_cell_line_name"].dropna())
unmatched_by_name = hpa_names_all - matched_by_name

# these unmatched names → look them up in HPA Desc → get CVCL → check if CVCL is in Sample Info RRID
hpa_desc_lookup = hpa_desc.set_index("Cell line")["Cellosaurus ID"].to_dict()
rrid_set = set(sample_info["RRID"].dropna())

resolved_via_cvcl = 0
still_unresolved  = []
for name in unmatched_by_name:
    cvcl = hpa_desc_lookup.get(name)
    if cvcl and cvcl in rrid_set:
        resolved_via_cvcl += 1
    else:
        still_unresolved.append((name, cvcl))

print(f"HPA names unmatched by name:          {len(unmatched_by_name):,}")
print(f"Resolved via HPA Desc CVCL → RRID:   {resolved_via_cvcl:,}")
print(f"Still unresolved:                     {len(still_unresolved):,}")
print(f"\nSample still unresolved (name, CVCL):")
for name, cvcl in sorted(still_unresolved)[:10]:
    print(f"  {name:<25} CVCL: {cvcl}")

HPA names unmatched by name:          281
Resolved via HPA Desc CVCL → RRID:   179
Still unresolved:                     102

Sample still unresolved (name, CVCL):
  537-mel                   CVCL: CVCL_8052
  624-mel                   CVCL: CVCL_8054
  888-mel                   CVCL: CVCL_4632
  AF22                      CVCL: None
  ASC2telo differentiated   CVCL: None
  ASC52telo                 CVCL: CVCL_U602
  BEWO                      CVCL: CVCL_0044
  BJ [Human fibroblast]     CVCL: CVCL_3653
  BJ hTERT+ SV40 Large T+   CVCL: None
  BJ hTERT+ SV40 Large T+ RasG12V CVCL: None


In [44]:
# Fix leading/trailing whitespace and recheck
geo_info["cell_line_trimmed"] = geo_info["cell_line"].str.strip()
geo_names_trimmed = set(geo_info["cell_line_trimmed"].dropna())

si_names     = set(sample_info["cell_line_name"].dropna())
si_stripped  = set(sample_info["stripped_cell_line_name"].dropna())

# also try uppercasing both sides
geo_upper    = set(s.upper() for s in geo_names_trimmed)
si_upper     = set(s.upper() for s in si_names) | set(s.upper() for s in si_stripped)

print("GEO name matching after whitespace fix:")
print(f"  Exact match:              {len(geo_names_trimmed & si_names):,}")
print(f"  Stripped match:           {len(geo_names_trimmed & si_stripped):,}")
print(f"  Case-insensitive match:  {len(geo_upper & si_upper):,}")

still_unmatched = geo_names_trimmed - si_names - si_stripped
print(f"  Still unmatched:          {len(still_unmatched):,}")
print(f"  Sample: {sorted(still_unmatched)[:15]}")

# how many of the unmatched have a CVCL in geo_info that IS in sample_info RRID?
rrid_set = set(sample_info["RRID"].dropna())
geo_with_cvcl = geo_info[geo_info["cell_line_trimmed"].isin(still_unmatched)]
geo_cvcl_resolvable = set(geo_with_cvcl["Cellosaurus_ID"].dropna()) & rrid_set
print(f"\n  Unmatched names but CVCL exists in RRID: {len(geo_cvcl_resolvable):,}")

GEO name matching after whitespace fix:
  Exact match:              201
  Stripped match:           497
  Case-insensitive match:  586
  Still unmatched:          351
  Sample: ['1A6', '537MEL', '624MEL', '7860', '888MEL', '928MEL', '977', 'A02', 'A04', 'A06', 'A11', 'A13', 'A15', 'A3_Kawakami', 'A4_Fukada']

  Unmatched names but CVCL exists in RRID: 94


In [45]:
# Classify the 102 unresolved HPA names
has_cvcl_not_in_rrid = []
no_cvcl              = []
has_brackets_or_plus = []

for name, cvcl in still_unresolved:
    if cvcl is None or pd.isna(cvcl):
        no_cvcl.append(name)
    elif cvcl not in rrid_set:
        has_cvcl_not_in_rrid.append((name, cvcl))
    # check for formatting issues
    if "[" in name or "+" in name or "(" in name:
        has_brackets_or_plus.append(name)

print(f"102 unresolved HPA names breakdown:")
print(f"  Has CVCL but NOT in Sample Info RRID:  {len(has_cvcl_not_in_rrid):,}")
print(f"  No CVCL at all:                        {len(no_cvcl):,}")
print(f"  Has brackets/plus (formatting issue):  {len(has_brackets_or_plus):,}")

print(f"\n--- Has CVCL but not in RRID (HPA-only cell lines) ---")
for name, cvcl in sorted(has_cvcl_not_in_rrid)[:15]:
    print(f"  {name:<35} {cvcl}")

print(f"\n--- No CVCL (engineered or uncatalogued) ---")
for name in sorted(no_cvcl)[:15]:
    print(f"  {name}")

# Can any bracket/plus names be rescued by stripping qualifiers?
print(f"\n--- Bracket/plus names — try base name extraction ---")
import re
for name in sorted(has_brackets_or_plus):
    base = re.split(r"[\[\(+]", name)[0].strip()
    in_si  = base in si_names or base in si_stripped
    in_cel = base in set(cellosaurus["Identifier (cell line name)"].dropna())
    print(f"  {name:<40} base='{base}'  in SI: {in_si}  in Cello: {in_cel}")

102 unresolved HPA names breakdown:
  Has CVCL but NOT in Sample Info RRID:  94
  No CVCL at all:                        8
  Has brackets/plus (formatting issue):  3

--- Has CVCL but not in RRID (HPA-only cell lines) ---
  537-mel                             CVCL_8052
  624-mel                             CVCL_8054
  888-mel                             CVCL_4632
  ASC52telo                           CVCL_U602
  BEWO                                CVCL_0044
  BJ [Human fibroblast]               CVCL_3653
  BJAB                                CVCL_5711
  C170                                CVCL_S959
  CCRF-SB                             CVCL_1860
  COLO 206F                           CVCL_1988
  COLO 320DM                          CVCL_0219
  COLO 699                            CVCL_1992
  COR-L26                             CVCL_2410
  COV413B                             CVCL_2423
  DEOC-1                              CVCL_4161

--- No CVCL (engineered or uncatalogued) ---
  AF22
  ASC